In [5]:
import pandas as pd

fake = pd.read_csv('/content/Fake.csv', on_bad_lines='skip', engine='python')
true = pd.read_csv('/content/True.csv', on_bad_lines='skip', engine='python')

print(fake.shape)
print(true.shape)

(11057, 4)
(11785, 4)


In [6]:
# Add labels
fake["label"] = 0
true["label"] = 1

# Combine datasets
df = pd.concat([fake, true], axis=0)

# Shuffle dataset
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Check result
print(df.shape)
df.head()

(22842, 5)


,title,text,subject,date,label
0,McCain CALLS OUT Trump: Show Us The Evidence!,Donald Trump screwed himself over by letting h...,News,"March 7, 2017",0
1,"Factbox: Trump on Twitter (July 9) - G20, Puti...",The following statements were posted to the ve...,politicsNews,"July 10, 2017",1
2,Republican Sandoval withdraws as possible Supr...,WASHINGTON (Reuters) - Nevada Governor Brian S...,politicsNews,"February 25, 2016",1
3,Trump administration releases rules on disclos...,WASHINGTON (Reuters) - The Trump administratio...,politicsNews,"November 15, 2017",1
4,BOOM! CONSUMER CONFIDENCE SOARS To Level Ameri...,Prior to the 2016 election (Oct 16) consumer ...,politics,"Oct 15, 2017",0


In [7]:
import re
import nltk
from nltk.corpus import stopwords

# Download stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

# Combine title + text
df["content"] = df["title"] + " " + df["text"]

# Clean function
def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r'[^a-zA-Z]', ' ', text)  # remove punctuation/numbers
    words = text.split()
    words = [word for word in words if word not in stop_words]  # remove stopwords
    return " ".join(words)

# Apply cleaning
df["content"] = df["content"].apply(clean_text)

# Check result
df[["content", "label"]].head()

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


,content,label
0,mccain calls trump show us evidence donald tru...,0
1,factbox trump twitter july g putin syria follo...,1
2,republican sandoval withdraws possible supreme...,1
3,trump administration releases rules disclosing...,1
4,boom consumer confidence soars level americans...,0


In [8]:
from sklearn.model_selection import train_test_split

X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(18273,) (4569,)


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

print(X_train.shape)

(18273, 5000)


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

# Initialize models
lr = LogisticRegression()
nb = MultinomialNB()
svm = LinearSVC()

# Train models
lr.fit(X_train, y_train)
nb.fit(X_train, y_train)
svm.fit(X_train, y_train)

LinearSVC()

In [11]:
from sklearn.metrics import accuracy_score, classification_report

models = {
    "Logistic Regression": lr,
    "Naive Bayes": nb,
    "SVM": svm
}

for name, model in models.items():
    y_pred = model.predict(X_test)

    print("\n==============================")
    print(name)
    print("==============================")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))


Logistic Regression
Accuracy: 0.9897132851827534
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      2245
           1       0.99      0.99      0.99      2324

    accuracy                           0.99      4569
   macro avg       0.99      0.99      0.99      4569
weighted avg       0.99      0.99      0.99      4569


Naive Bayes
Accuracy: 0.9446268330050339
              precision    recall  f1-score   support

           0       0.95      0.94      0.94      2245
           1       0.94      0.95      0.95      2324

    accuracy                           0.94      4569
   macro avg       0.94      0.94      0.94      4569
weighted avg       0.94      0.94      0.94      4569


SVM
Accuracy: 0.9932151455460714
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      2245
           1       0.99      0.99      0.99      2324

    accuracy                           0.99      4569
  

In [12]:
import pickle

# Save SVM model
pickle.dump(svm, open("best_model.pkl", "wb"))

# Save vectorizer
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

In [15]:
import numpy as np

y_pred = svm.predict(X_test)

print("Predicted counts:", np.bincount(y_pred))
print("Actual counts:", np.bincount(y_test))

Predicted counts: [2240 2329]
Actual counts: [2245 2324]


In [17]:
def predict_news(text):
    cleaned = clean_text(text)
    print("Cleaned text:", cleaned[:100])  # debug

    vector = vectorizer.transform([cleaned])
    prediction = svm.predict(vector)

    return "Real News" if prediction[0] == 1 else "Fake News"

In [18]:
print(predict_news("The United States government announced a new economic policy after discussions in the senate and parliament."))

Cleaned text: united states government announced new economic policy discussions senate parliament
Real News


In [20]:
print(predict_news("Breaking shocking truth revealed you won't believe what government is hiding from you"))

Cleaned text: breaking shocking truth revealed believe government hiding
Fake News
